# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Pretty print basic metadata
metadata = dataset.metadata
print(f"Title: {getattr(metadata, 'name', '')}")
print(f"Description: {getattr(metadata, 'description', '')}")
print(f"Version: {getattr(metadata, 'version', '')}")
print(f"License: {getattr(metadata, 'license', '')}")


## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

Let's inspect all record sets in the dataset. Each will be referenced by their `@id`.

In [ ]:
# List available record sets via their @id and name
record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record set(s)\n")
for i, rs in enumerate(record_sets):
    print(f"[{i}] RecordSet @id: {rs['@id']}")
    print(f"    Name: {rs.get('name','')}")
    print(f"    Description: {rs.get('description','')}")
    # List fields/columns in this record set by @id
    if 'fields' in rs:
        print("    Fields:")
        for field in rs['fields']:
            print(f"      - {field['@id']}: {field.get('name','')} ({field.get('dataType','')})")
    if 'columns' in rs:
        print("    Columns:")
        for column in rs['columns']:
            print(f"      - {column['@id']}: {column.get('name','')} ({column.get('dataType','')})")
    print()
if not record_sets:
    print('No record sets found in metadata. Please inspect dataset.metadata for structure.')


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

**Instructions:**

- Identify a suitable record set `@id` from above. 
- Use that `@id` value to load data and inspect columns by their `@id`. 
- For this dataset, the relevant record set is likely the main protein abundance table. If you see only one, use that; otherwise, choose one to explore.

Below we attempt to extract all record sets available.

In [ ]:
# We will automatically extract all record sets referenced by their @id
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records for: {rs_id}")
            print(f"Columns (by field/column @id) for {rs_id}")
            print(dataframes[rs_id].columns.tolist())
            print(dataframes[rs_id].head(), '\n')
        else:
            print(f"No records found for record set {rs_id}.")
    except Exception as e:
        print(f"Failed to load records from {rs_id}: {e}")

if dataframes:
    # Choose the first record set for demonstration
    main_record_set_id = list(dataframes.keys())[0]
else:
    main_record_set_id = None


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- Select a numeric field (`@id`).
- Apply filtering and normalization.
- Attempt grouping by another field if available.

In [ ]:
if main_record_set_id is not None:
    df = dataframes[main_record_set_id].copy()
    print(f"Columns in {main_record_set_id}:\n", df.columns.tolist())
    # Heuristically pick a numeric field (by name or index)
    # E.g. 'cr:coverage_percent' or similar
    numeric_candidate_fields = [col for col in df.columns if df[col].dtype in [float, int] or pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_candidate_fields:
        # Try to coerce any suitable looking column that is likely numeric (e.g. contains 'mw', 'abundance', 'count', 'percent')
        for col in df.columns:
            if any(t in col.lower() for t in ['abundance', 'mw', 'count', 'percent', 'number', 'intensity']):
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except Exception:
                    pass
        numeric_candidate_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Identified numeric fields: {numeric_candidate_fields}")
    if numeric_candidate_fields:
        numeric_field = numeric_candidate_fields[0] # Use the first for demonstration
        threshold = df[numeric_field].mean() if pd.notna(df[numeric_field].mean()) else 0
        print(f"Applying filter: {numeric_field} > {threshold:.2f}")
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field if present (e.g. 'cr:sample', 'cr:modification', etc)
        group_candidate_fields = [col for col in df.columns if col != numeric_field and df[col].nunique() < 10 and pd.api.types.is_object_dtype(df[col])]
        if group_candidate_fields:
            group_field = group_candidate_fields[0]
            print(f"Grouping by {group_field}:")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(grouped_df.head())
    else:
        print("No numeric field found for analysis.")
else:
    print("No main record set loaded; cannot proceed with EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For illustration, we plot the histogram of the main numeric field and a boxplot if possible.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if main_record_set_id is not None and numeric_candidate_fields:
    plt.figure(figsize=(8, 4))
    df = dataframes[main_record_set_id]
    plt.hist(df[numeric_field].dropna(), bins=30, color='skyblue', edgecolor='black')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_candidate_fields:
        plt.figure(figsize=(8, 4))
        df.boxplot(column=numeric_field, by=group_candidate_fields[0], grid=False)
        plt.title(f'{numeric_field} by {group_candidate_fields[0]}')
        plt.suptitle('')
        plt.xlabel(group_candidate_fields[0])
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No data available for visualization.")


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the [FAIR^2: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset via its Croissant schema.
- Inspected all record sets, their fields and columns using their `@id` identifiers.
- Loaded the main available record set into a DataFrame.
- Performed basic EDA: filter by a numeric field, normalize, and grouped by available categorical fields.
- Plotted data distribution and explored relationships between fields.

Explore further using field `@id`s for detailed protein, modification, or quantitation analysis as appropriate for experimental needs.